In [2]:
# ─── 0. Environment Check ───────────────────────────────────────────────────
import torch
print(torch.cuda.is_available())
!nvidia-smi
!pip install -U transformers datasets accelerate sentencepiece sacrebleu -q
# ─── 1. Imports ─────────────────────────────────────────────────────────────
import numpy as np
import sacrebleu
import transformers
from datasets import load_dataset
from transformers import (
    MarianTokenizer,
    MarianMTModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
print(transformers.__version__)
# ─── 2. Device Setup ─────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
# ─── 3. Dataset & Model ──────────────────────────────────────────────────────
dataset    = load_dataset("opus100", "en-fr")
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model     = MarianMTModel.from_pretrained(model_name)
model.to(device)
# ─── 4. Preprocessing ────────────────────────────────────────────────────────
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 128
def preprocess_function(examples):
    inputs  = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
tokenized_datasets = dataset.map(preprocess_function, batched=True)
data_collator      = DataCollatorForSeq2Seq(tokenizer, model=model)
# ─── 5. Metrics ──────────────────────────────────────────────────────────────
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": bleu.score}
# ─── 6. Training ─────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./nmt_en_fr",
    learning_rate               = 2e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    num_train_epochs            = 1,
    predict_with_generate       = True,
    logging_steps               = 100,
    eval_strategy               = "steps",
    eval_steps                  = 500,
)
trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized_datasets["train"].select(range(5000)),
    eval_dataset     = tokenized_datasets["validation"].select(range(1000)),
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)
trainer.train()
# ─── 7. Evaluation ───────────────────────────────────────────────────────────
trainer.predict(tokenized_datasets["validation"].select(range(300)))
# ─── 8. Inference ────────────────────────────────────────────────────────────
def translate(text):
    inputs  = tokenizer(text, return_tensors="pt", padding=True).to(device)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
test_sentences = [
    "How are you?",
    "Machine learning is powerful.",
]
for sentence in test_sentences:
    print("EN:", sentence)
    print("FR:", translate(sentence))
    print()

True
Mon Mar 30 08:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             30W /   70W |     399MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Bleu
500,0.628296,0.809346,27.002589
625,0.635280,0.802550,27.260766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

EN: How are you?
FR: Comment vaz-vous ?

EN: Machine learning is powerful.
FR: La machine learning est pécaine.

